In [5]:
import sys
import os

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

sys.path.append(os.getcwd())

from src.models import CNN
from src.train import train
from src.dataset import HAM10000Dataset, get_train_transforms, get_eval_transforms, compute_class_weights, IMAGENET_MEAN, IMAGENET_STD, DX_TO_IDX, IDX_TO_DX
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader

Read in training and validation data, initialize HAM10000Dataset objects for each

In [ ]:
train_df = pd.read_csv('data/splits/train.csv')
val_df = pd.read_csv('data/splits/val.csv')

image_dir = os.path.join(os.getcwd(), 'data', 'HAM10000_images')

train_dataset = HAM10000Dataset(train_df, image_dir, get_train_transforms())
val_dataset = HAM10000Dataset(val_df, image_dir, get_eval_transforms())

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, num_workers=2)

Create CNN

In [7]:
cnn = CNN(
   in_channels=3,
   num_classes=7,
   n_blocks=4,
   n_layers_per_block=6,
   channels_l0=16,
   stem_kernel_size=11,
   stem_stride=2,
   block_stride=2,
   channel_growth_factor=2,
   dropout_p=0.3 
)

Construct loss function

In [8]:
class_weights = compute_class_weights(train_df)

loss_func = torch.nn.CrossEntropyLoss(weight=class_weights)

Construct Optimizer

In [9]:
lr = 1e-3
momentum = 0.9
adaptive_learning_rate = 0.999
betas = (momentum, adaptive_learning_rate)
optimizer = torch.optim.AdamW(cnn.parameters(), lr=lr, betas=betas)

Train CNN

In [10]:
cnn = train(
    model=cnn,
    model_name="baseline_cnn",
    train_loader=train_loader,
    val_loader=val_loader,
    loss_func=loss_func,
    optimizer=optimizer,
    num_epoch=50
)

CUDA not available, using CPU


KeyError: Caught KeyError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "c:\Users\scott\miniconda3\envs\ai-in-healthcare\Lib\site-packages\pandas\core\indexes\base.py", line 3641, in get_loc
    return self._engine.get_loc(casted_key)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "pandas/_libs/index.pyx", line 168, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/index.pyx", line 176, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/index.pyx", line 583, in pandas._libs.index.StringObjectEngine._check_type
KeyError: 3186

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "c:\Users\scott\miniconda3\envs\ai-in-healthcare\Lib\site-packages\torch\utils\data\_utils\worker.py", line 374, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\scott\miniconda3\envs\ai-in-healthcare\Lib\s